# grspin quick start

Relax classical spins with the general-relativistic spin-spin interaction on a Poisson point set, measure the pair correlations and compare with a control interaction. The run below is small (about a minute); the production plan is in `docs/SIMULATION_PLAN.md`.

Definitions of the model and the observables: `docs/METHODS.md`.

In [ ]:
%matplotlib inline
from IPython.display import display

from grspin import SimulationConfig
from grspin.analysis import EnsembleData, paired_difference
from grspin.ensemble import run_ensemble
from grspin.fit import FitResult, fit_ensemble
from grspin.plotting import figure_C, figure_compare, figure_overview

## 1. Configure

`n_bins=80` radial bins on `[0, 4] r_nn` are stored; the analysis re-bins by 2. Periodic boundaries and a cutoff of `10 r_nn` are the defaults for `N >= 2600` (for smaller boxes the cutoff is reduced to `L/2` automatically).

In [ ]:
config = SimulationConfig(
    n_particles=2000,
    n_sweeps=100,
    n_bins=80,
    n_realizations=16,
    base_seed=1000,
)
print(f"box L = {config.box:.2f}, r_nn = {config.r_nn:.4f}, cutoff = {config.r_cut_nn_resolved:.2f} r_nn,"
      f" {config.neighbors_per_particle:.0f} neighbours per spin")
config.estimate_cost()

## 2. Run an ensemble

Every realization is written to disk as soon as it finishes, so the cell can be re-run (it resumes) or the ensemble extended by raising `n_realizations`. With `n_workers > 1` the realizations run in parallel processes.

In [ ]:
info = run_ensemble(config, "results/quickstart/gr", n_workers=4, progress=False)
print(f"{info.n_computed} realizations computed in {info.elapsed_s:.1f} s")

## 3. Look at the result

`EnsembleData` holds the binned sums of all realizations. `table()` gives ensemble means with standard errors over realizations (as in the manuscript); `landmarks()` gives model-independent features with bootstrap errors.

In [ ]:
data = EnsembleData.load("results/quickstart/gr")
table = data.table(rebin=2)
display(table[["x_center", "mean_C_final", "sem_C_final", "mean_Q_final", "mean_A_final", "mean_B_final"]].head(12))

In [ ]:
marks = data.landmarks(rebin=2, n_boot=100)
for key in ("x_zero_C", "x_peak_C", "C_peak", "Q_min", "x_min_Q"):
    info = marks[key]
    print(f"{key:10s} = {info['value']:.3f} +- {info['err']:.3f}")

In [ ]:
fit = fit_ensemble(data, rebin=2, n_boot=0)["fit"]  # phenomenological; see docs/METHODS.md, section 7
figure_overview(data, rebin=2, fit=FitResult(**fit) if fit["success"] else None)

## 4. A control with the same geometries

The same seeds give the same positions and initial directions, so changing only the coupling yields *paired* differences. `dipolar` is the sign-reversed tensor (a dipolar magnet), `isotropic_afm` an isotropic `1/r^3` antiferromagnet.

In [ ]:
run_ensemble(config.replace(coupling="dipolar"), "results/quickstart/dipolar", n_workers=4, progress=False)
run_ensemble(config.replace(coupling="isotropic_afm"), "results/quickstart/afm", n_workers=4, progress=False)
runs = {
    "GR tensor": data,
    "dipolar (sign-reversed)": EnsembleData.load("results/quickstart/dipolar"),
    "isotropic AFM": EnsembleData.load("results/quickstart/afm"),
}
figure_compare(runs, ("C", "Q", "A"), rebin=2)

In [ ]:
diff = paired_difference(runs["GR tensor"], runs["isotropic AFM"], "C", rebin=2)
display(diff[["x_center", "diff", "sem", "abs_diff_over_sem"]].iloc[4:14])

## Next steps

* `grspin plan list` shows the production plan; `grspin plan run all --smoke` tests the whole pipeline in miniature.
* `grspin analyze <run>` writes tables, landmarks, fit and figures; `grspin compare` does the overlays above.
* On macOS and Windows scripts that use `n_workers > 1` need an `if __name__ == "__main__":` guard (notebooks do not).